
# Stage 4 — Presentation Graphs & Tables

Consolidates every high-yield analysis discussed for presenting Stage 4's
results into one standalone notebook: the saliency-experiment charts
(threshold sweep, an exemplar saliency timeline, an animated video+saliency
"playthrough", per-phase rescue impact, hybrid-vs-global-threshold
coverage) and the four-reel comparison charts (summary-by-variant, Reel 3
vs Reel 4, a temporal phase-coverage checklist, a per-video best-reel
ranking, inter-reel Jaccard overlap).

**This notebook only reads CSVs that `stage4_highlight_reels.ipynb`
already wrote to disk** (plus, for the one animated chart in Section 2.2b,
the audio-free video files it already wrote to `no_audio_videos/`, via
`ffmpeg`/`ffprobe`). It never recomputes saliency, never calls a vision or
selection LLM, and never loads the phase-recognition model or a GPU — so
it's fast, cheap to re-run, and immune to the kernel-restart `NameError`s
that plague a long-lived pipeline notebook: every input comes from disk,
not from another notebook's in-memory state.

Run this any time after `stage4_highlight_reels.ipynb` has executed
Sections 3 (saliency) through 9 (metrics) at least once. Each graph is
saved as a PNG into its own numbered subfolder under
`stage4/presentation/`, and every summary table used across those graphs
is also written into one Word document,
`stage4/presentation/tables/stage4_summary_tables.docx`.


## 0. Setup


In [ ]:

!pip install -q python-docx


In [ ]:

from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from docx import Document
from docx.shared import Pt, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH



### Config

Point `project_root` at the same Drive folder `stage4_highlight_reels.ipynb`
used. Every path under `DIRS` is read-only except `DIRS["presentation"]`
and the numbered subfolders under it, which this notebook owns.


In [ ]:

@dataclass
class Config:
    project_root: str = "/content/drive/MyDrive/hypospadias_stage2"
    videos: list = field(default_factory=lambda: ["1", "2", "4", "5", "6", "7", "8"])
    llm_backends: list = field(default_factory=lambda: ["chatgpt", "claude", "ollama"])

    # Must match stage4_highlight_reels.ipynb's CFG for these same fields —
    # used only to find the right saved CSVs and draw reference lines, never
    # to recompute anything from raw features.
    saliency_thresholds: list = field(default_factory=lambda: [0.0, 0.1, 0.2, 0.3, 0.4])
    saliency_operating_threshold: float = 0.1
    per_phase_rescue_percentile: float = 80.0
    min_segment_s: float = 1.0
    reel_target_min_s: float = 120.0
    reel_target_max_s: float = 300.0
    reel_leeway_s: float = 30.0
    playback_speed: float = 2.0


CFG = Config()
ROOT = Path(CFG.project_root)
RAW_DURATION_RANGE = (
    (CFG.reel_target_min_s - CFG.reel_leeway_s) * CFG.playback_speed,
    (CFG.reel_target_max_s + CFG.reel_leeway_s) * CFG.playback_speed,
)

DIRS = {
    "manifest": ROOT / "manifest",
    "no_audio_videos": ROOT / "stage4" / "no_audio_videos",
    "saliency": ROOT / "stage4" / "saliency",
    "candidates": ROOT / "stage4" / "candidates",
    "selections": ROOT / "stage4" / "selections",
    "metrics": ROOT / "stage4" / "metrics",
    "presentation": ROOT / "stage4" / "presentation",
}

GRAPH_DIRS = {
    name: DIRS["presentation"] / name
    for name in [
        "01_threshold_sweep", "02_saliency_timeline", "02b_saliency_playthrough",
        "03_rescue_impact", "04_hybrid_vs_global_coverage", "05_summary_by_variant",
        "06_reel3_vs_reel4_dumbbell", "07_reel3_vs_reel4_head_to_head",
        "08_phase_coverage_checklist", "09_best_reel_ranking",
        "10_inter_reel_jaccard", "tables",
    ]
}
for d in GRAPH_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print(f"Presentation assets will be saved under: {DIRS['presentation']}")


### Style — same palette used throughout this project's charts


In [ ]:

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
INK, SECONDARY_INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
GOOD, CRITICAL = "#0ca30c", "#d03b3b"
BLUE_SEQUENTIAL = LinearSegmentedColormap.from_list(
    "blue_sequential", ["#f0efec", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]
)

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": SECONDARY_INK, "ytick.color": SECONDARY_INK,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
})



## 1. Load everything from disk

Every loader below either reads a CSV `stage4_highlight_reels.ipynb`
already wrote, or (for the global-threshold-only saliency pool, which is
only ever built for comparison and may not have been saved) rebuilds it
cheaply from the saved per-video saliency signal — no model, no GPU.


In [ ]:

def load_manifest() -> pd.DataFrame:
    path = DIRS["manifest"] / "frames_manifest.csv"
    if not path.exists():
        raise FileNotFoundError(f"{path} missing — run Stage 2's build_full_manifest() first.")
    return pd.read_csv(path, dtype={"video_id": str})


def load_saliency_smoothed() -> pd.DataFrame:
    parts = []
    for video_id in CFG.videos:
        path = DIRS["saliency"] / f"{video_id}_smoothed.csv"
        if not path.exists():
            print(f"  missing {path} — skipping video {video_id}")
            continue
        parts.append(pd.read_csv(path, dtype={"video_id": str}))
    if not parts:
        raise FileNotFoundError("No *_smoothed.csv found — run Section 3 of stage4_highlight_reels.ipynb first.")
    return pd.concat(parts, ignore_index=True)


def load_threshold_sweep() -> pd.DataFrame:
    path = DIRS["saliency"] / "threshold_sweep_summary.csv"
    if not path.exists():
        raise FileNotFoundError(f"{path} missing — run sweep_thresholds() in stage4_highlight_reels.ipynb first.")
    return pd.read_csv(path, dtype={"video_id": str})


def load_hybrid_pool() -> pd.DataFrame:
    matches = sorted(DIRS["candidates"].glob(
        f"saliency_pool_hybrid_tau{CFG.saliency_operating_threshold}_p*.csv"))
    if not matches:
        raise FileNotFoundError(
            f"No saliency_pool_hybrid_tau{CFG.saliency_operating_threshold}_p*.csv in {DIRS['candidates']} — "
            "run build_candidate_pool_hybrid() in stage4_highlight_reels.ipynb first."
        )
    return pd.read_csv(matches[-1], dtype={"video_id": str})


def frame_diffs_to_segments(mask: np.ndarray):
    segments = []
    start = None
    for i, v in enumerate(mask):
        if v and start is None:
            start = i
        elif not v and start is not None:
            segments.append((start, i))
            start = None
    if start is not None:
        segments.append((start, len(mask)))
    return segments


def extract_segments(df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    df = df.sort_values("frame_idx").reset_index(drop=True)
    mask = (df.saliency_smooth >= threshold).values
    rows = []
    for start, end in frame_diffs_to_segments(mask):
        seg = df.iloc[start:end]
        duration = seg.timestamp.iloc[-1] - seg.timestamp.iloc[0]
        if duration < CFG.min_segment_s:
            continue
        phase_mode = seg.predicted_phase.mode()
        rows.append({
            "video_id": seg.video_id.iloc[0],
            "start_ts": seg.timestamp.iloc[0], "end_ts": seg.timestamp.iloc[-1],
            "duration_s": duration,
            "phase": phase_mode.iloc[0] if len(phase_mode) else seg.predicted_phase.iloc[0],
            "mean_saliency": seg.saliency_smooth.mean(), "peak_saliency": seg.saliency_smooth.max(),
        })
    return pd.DataFrame(rows)


def load_or_build_global_pool(saliency: pd.DataFrame) -> pd.DataFrame:
    path = DIRS["candidates"] / f"saliency_pool_tau{CFG.saliency_operating_threshold}.csv"
    if path.exists():
        return pd.read_csv(path, dtype={"video_id": str})
    print(f"  {path} not found — rebuilding the global-threshold-only pool from saved saliency (comparison only).")
    parts = [extract_segments(g, CFG.saliency_operating_threshold) for _, g in saliency.groupby("video_id")]
    return pd.concat(parts, ignore_index=True)


def get_manifest_positions(manifest: pd.DataFrame, video_id: str, start_ts: float, end_ts: float) -> np.ndarray:
    vm = manifest[manifest.video_id == video_id].sort_values("frame_idx").reset_index(drop=True)
    mask = (vm.timestamp >= start_ts) & (vm.timestamp <= end_ts)
    return np.flatnonzero(mask.values)


def frame_coverage_mask(clips: pd.DataFrame, manifest: pd.DataFrame, video_id: str) -> np.ndarray:
    n = (manifest.video_id == video_id).sum()
    mask = np.zeros(n, dtype=bool)
    for row in clips.itertuples():
        idx = get_manifest_positions(manifest, video_id, row.start_ts, row.end_ts)
        mask[idx[idx < n]] = True
    return mask


def temporal_coverage_per_phase(clips: pd.DataFrame, manifest: pd.DataFrame, video_id: str) -> dict:
    '''Fraction of each ground-truth phase's own frames that fall inside any
    of this reel's selected clip spans -- true frame-level temporal overlap,
    not just "does the reel contain a clip labelled with this phase name".'''
    vm = manifest[manifest.video_id == video_id].sort_values("frame_idx").reset_index(drop=True)
    mask = frame_coverage_mask(clips, manifest, video_id) if len(clips) else np.zeros(len(vm), dtype=bool)
    out = {}
    for phase, group in vm.groupby("phase"):
        idx = group.index.values
        idx = idx[idx < len(mask)]
        out[phase] = mask[idx].mean() if len(idx) else float("nan")
    return out


def phase_coverage(clips: pd.DataFrame, manifest: pd.DataFrame, video_id: str) -> float:
    phases_in_video = set(manifest[manifest.video_id == video_id].phase.unique())
    phases_in_reel = set(clips.phase.unique()) if len(clips) else set()
    return len(phases_in_reel & phases_in_video) / len(phases_in_video) if phases_in_video else float("nan")


def jaccard_similarity(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    union = (mask_a | mask_b).sum()
    return (mask_a & mask_b).sum() / union if union else float("nan")


def get_phase_order(manifest: pd.DataFrame) -> list:
    '''Ground-truth chronological order: each phase ranked by the earliest
    timestamp it's ever seen at, across all videos.'''
    return list(manifest.groupby("phase").timestamp.min().sort_values().index)


def load_reel_clips(video_id: str, reel_variant: str) -> pd.DataFrame:
    '''reel_variant like "reel1", "reel2_claude", "reel3_claude", "reel4_claude".'''
    if reel_variant == "reel1":
        path = DIRS["selections"] / f"reel1_{video_id}.csv"
    else:
        name, backend = reel_variant.split("_", 1)
        path = DIRS["selections"] / f"{name}_{backend}_{video_id}.csv"
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, dtype={"video_id": str})


def load_all_reels() -> dict:
    '''{(video_id, reel_variant): clips_df} for every reel1/2/3/4 x backend x
    video CSV actually present on disk -- silently skips combinations that
    were never run, same as the main notebook's own build_reelN_all() loops.'''
    all_reels = {}
    for video_id in CFG.videos:
        clips = load_reel_clips(video_id, "reel1")
        if len(clips):
            all_reels[(video_id, "reel1")] = clips
    for reel_name in ["reel2", "reel3", "reel4"]:
        for backend in CFG.llm_backends:
            for video_id in CFG.videos:
                clips = load_reel_clips(video_id, f"{reel_name}_{backend}")
                if len(clips):
                    all_reels[(video_id, f"{reel_name}_{backend}")] = clips
    return all_reels


def load_metrics() -> pd.DataFrame:
    path = DIRS["metrics"] / "reel_metrics.csv"
    if not path.exists():
        raise FileNotFoundError(f"{path} missing — run run_all_metrics() in stage4_highlight_reels.ipynb first.")
    return pd.read_csv(path, dtype={"video_id": str})


In [ ]:

manifest = load_manifest()
saliency = load_saliency_smoothed()
threshold_sweep = load_threshold_sweep()
saliency_candidates = load_hybrid_pool()
saliency_candidates_global = load_or_build_global_pool(saliency)
all_reels = load_all_reels()
metrics = load_metrics()

REEL3_VARIANT = next((v for (_, v) in all_reels if v.startswith("reel3_")), None)
REEL4_VARIANT = next((v for (_, v) in all_reels if v.startswith("reel4_")), None)

print(f"{len(manifest)} manifest rows, {len(saliency)} saliency rows, "
      f"{len(all_reels)} (video, reel_variant) combinations loaded, {len(metrics)} metric rows")
print(f"Reel 3 variant: {REEL3_VARIANT!r}, Reel 4 variant: {REEL4_VARIANT!r}")



A consistent color per reel variant, reused across every chart below so
the same reel always reads as the same color throughout the deck.


In [ ]:

VARIANT_ORDER = sorted(metrics.reel_variant.unique(), key=lambda v: (v.split("_")[0], v))
VARIANT_COLOR = {variant: PALETTE[i % len(PALETTE)] for i, variant in enumerate(VARIANT_ORDER)}
print(VARIANT_COLOR)



## 2. Saliency experiments

### 2.1 Threshold sweep — why tau={CFG.saliency_operating_threshold}

Every video must clear the raw-duration floor at the chosen tau; this is
the direct evidence for that choice, not an eyeballed number.


In [ ]:

def plot_threshold_sweep(threshold_sweep: pd.DataFrame, operating_tau: float,
                          duration_floor_s: float, duration_ceiling_s: float, out_path):
    videos = sorted(threshold_sweep.video_id.unique(), key=lambda v: int(v))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    for i, video_id in enumerate(videos):
        vs = threshold_sweep[threshold_sweep.video_id == video_id].sort_values("threshold")
        color = PALETTE[i % len(PALETTE)]
        ax1.plot(vs.threshold, vs.total_duration_s / 60, marker="o", color=color,
                  label=f"Video {video_id}", linewidth=1.8)
        ax2.plot(vs.threshold, vs.n_segments, marker="o", color=color, linewidth=1.8)

    ax1.axhline(duration_floor_s / 60, color=CRITICAL, linestyle="--", linewidth=1.2, label="Raw duration floor")
    ax1.axhline(duration_ceiling_s / 60, color=MUTED, linestyle=":", linewidth=1.2)
    for ax in (ax1, ax2):
        ax.axvline(operating_tau, color=INK, linestyle="--", linewidth=1.3, alpha=0.7)
        ax.set_xlabel("Saliency threshold (tau)")
    ax1.text(operating_tau, ax1.get_ylim()[1] * 0.97, f"  tau={operating_tau}", fontsize=9, color=INK, va="top")

    ax1.set_ylabel("Total candidate duration (min)")
    ax1.set_title("Candidate duration vs. tau", fontsize=11, weight="bold")
    ax2.set_ylabel("Number of candidate segments")
    ax2.set_title("Segment count vs. tau", fontsize=11, weight="bold")
    handles, labels = ax1.get_legend_handles_labels()
    fig.suptitle("Threshold sweep — every video must clear the duration floor at the chosen tau",
                  fontsize=12, weight="bold")
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.02),
               ncol=4, fontsize=8, frameon=False)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor=SURFACE)
    plt.show()
    print(f"Saved {out_path}")


plot_threshold_sweep(threshold_sweep, CFG.saliency_operating_threshold,
                      RAW_DURATION_RANGE[0], RAW_DURATION_RANGE[1],
                      GRAPH_DIRS["01_threshold_sweep"] / "threshold_sweep.png")



### 2.2 Exemplar saliency timeline

Split into two focused charts per video rather than one crowded overlay:
one showing only the ground-truth phases (with a proper phase-color
legend), and one showing only the saliency signal and what got extracted
from it. Saved for every video; pick whichever is most representative for
the deck (Video 5, the MIP case, is a good default since it's the one
where the reel variants meaningfully diverge).


In [ ]:

from matplotlib.patches import Patch


def phase_to_segments(df: pd.DataFrame, phase_col: str, ts_col: str = "timestamp"):
    df = df.sort_values(ts_col).reset_index(drop=True)
    change = (df[phase_col] != df[phase_col].shift()).cumsum()
    segs = df.groupby(change).agg(phase=(phase_col, "first"), start_ts=(ts_col, "first"), end_ts=(ts_col, "last"))
    return segs.reset_index(drop=True)


def plot_ground_truth_timeline(video_id: str, manifest: pd.DataFrame, phase_color: dict, out_path):
    gt = manifest[manifest.video_id == video_id].sort_values("timestamp")
    gt_segs = phase_to_segments(gt, "phase")

    fig, ax = plt.subplots(figsize=(14, 3.2))
    phases_in_video = []
    for _, seg in gt_segs.iterrows():
        ax.axvspan(seg.start_ts / 60, seg.end_ts / 60, color=phase_color.get(seg.phase, MUTED),
                    alpha=0.5, linewidth=0)
        if seg.phase not in phases_in_video:
            phases_in_video.append(seg.phase)

    ax.set_xlim(gt.timestamp.min() / 60, gt.timestamp.max() / 60)
    ax.set_yticks([])
    ax.set_xlabel("Time (min)")
    ax.set_title(f"Video {video_id} — ground-truth phases", fontsize=11, weight="bold")

    handles = [Patch(facecolor=phase_color.get(p, MUTED), alpha=0.7, label=p) for p in phases_in_video]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.4),
              ncol=4, fontsize=7.5, frameon=False)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor=SURFACE)
    plt.close(fig)


def plot_saliency_extraction(video_id: str, smoothed: pd.DataFrame, candidates: pd.DataFrame,
                              operating_tau: float, out_path):
    sal = smoothed[smoothed.video_id == video_id].sort_values("timestamp")
    clips = candidates[candidates.video_id == video_id]

    fig, ax = plt.subplots(figsize=(14, 4))
    for _, clip in clips.iterrows():
        ax.axvspan(clip.start_ts / 60, clip.end_ts / 60, color=GOOD, alpha=0.35, linewidth=0)

    ax.plot(sal.timestamp / 60, sal.saliency_smooth, color=INK, linewidth=1.1, label="Smoothed saliency")
    ax.axhline(operating_tau, color=CRITICAL, linestyle="--", linewidth=1.2, label=f"tau={operating_tau}")

    handles, labels = ax.get_legend_handles_labels()
    handles.append(Patch(facecolor=GOOD, alpha=0.35, label="Extracted candidate segment"))
    ax.set_xlabel("Time (min)")
    ax.set_ylabel("Smoothed saliency (normalised)")
    ax.set_title(f"Video {video_id} — saliency signal and extracted candidates", fontsize=11, weight="bold")
    ax.legend(handles=handles, loc="upper right", fontsize=9, frameon=False)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor=SURFACE)
    plt.close(fig)


all_phases = sorted(manifest.phase.unique())
phase_color = {p: PALETTE[i % len(PALETTE)] for i, p in enumerate(all_phases)}

for video_id in CFG.videos:
    gt_path = GRAPH_DIRS["02_saliency_timeline"] / f"ground_truth_timeline_{video_id}.png"
    plot_ground_truth_timeline(video_id, manifest, phase_color, gt_path)
    print(f"Saved {gt_path}")

    sal_path = GRAPH_DIRS["02_saliency_timeline"] / f"saliency_extraction_{video_id}.png"
    plot_saliency_extraction(video_id, saliency, saliency_candidates,
                              CFG.saliency_operating_threshold, sal_path)
    print(f"Saved {sal_path}")

# Preview the recommended exemplar (Video 5) inline.
from IPython.display import Image, display
display(Image(str(GRAPH_DIRS["02_saliency_timeline"] / "ground_truth_timeline_5.png")))
display(Image(str(GRAPH_DIRS["02_saliency_timeline"] / "saliency_extraction_5.png")))



### 2.2b Saliency "playthrough" animation

An accelerated video+saliency animation for a slide: the source video
plays back heavily sped up on top while the saliency curve draws in
left-to-right underneath it, in sync, with a moving playhead. This
compresses an hour-plus of footage into a ~45-second clip — the same
accelerated-playback idea as the 2x-speed reels, just pushed much
further, purely for illustrating what the saliency signal means over
time. Saves an `.mp4` you can drop straight into PowerPoint or Google
Slides as a normal video object (no in-app "animation" needed — the
motion is baked into the file).

Adjust `target_duration_s` / `target_fps` below to trade render time
against smoothness; the defaults (45s at 15fps) render in a couple of
minutes and look smooth enough for a slide.


In [ ]:

import subprocess
import tempfile
import shutil

import matplotlib.animation as animation
import matplotlib.image as mpimg
from IPython.display import Video


def no_audio_path(video_id: str) -> Path:
    p = DIRS["no_audio_videos"] / f"{video_id}.mp4"
    if not p.exists():
        raise FileNotFoundError(f"{p} missing — run strip_audio_all_videos() in stage4_highlight_reels.ipynb first.")
    return p


def get_video_duration(video_id: str) -> float:
    src = no_audio_path(video_id)
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", str(src)],
        check=True, capture_output=True, text=True,
    )
    return float(result.stdout.strip())


def extract_frames_for_animation(video_id: str, target_duration_s: float, target_fps: float, tmp_dir: Path):
    '''Pulls one frame from the source video roughly every
    source_duration / (target_duration_s * target_fps) seconds, via a
    single ffmpeg pass (far faster than one subprocess call per frame).'''
    src = no_audio_path(video_id)
    source_duration = get_video_duration(video_id)
    n_target_frames = max(1, round(target_duration_s * target_fps))
    sample_fps = n_target_frames / source_duration

    pattern = tmp_dir / "frame_%05d.jpg"
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(src), "-vf", f"fps={sample_fps}", "-q:v", "3", str(pattern)],
        check=True, capture_output=True,
    )
    frame_paths = sorted(tmp_dir.glob("frame_*.jpg"))
    if not frame_paths:
        raise RuntimeError(f"ffmpeg produced no frames for video {video_id} — check {src} is a valid video.")
    frame_timestamps = [i / sample_fps for i in range(len(frame_paths))]
    return frame_paths, frame_timestamps, source_duration


def render_saliency_playthrough(video_id: str, smoothed: pd.DataFrame, manifest: pd.DataFrame,
                                 candidates: pd.DataFrame, operating_tau: float, phase_color: dict,
                                 out_path: Path, target_duration_s: float = 45, target_fps: float = 15):
    tmp_dir = Path(tempfile.mkdtemp(prefix=f"playthrough_{video_id}_"))
    try:
        frame_paths, frame_timestamps, source_duration = extract_frames_for_animation(
            video_id, target_duration_s, target_fps, tmp_dir)

        sal = smoothed[smoothed.video_id == video_id].sort_values("timestamp").reset_index(drop=True)
        gt = manifest[manifest.video_id == video_id].sort_values("timestamp")
        gt_segs = phase_to_segments(gt, "phase")
        clips = candidates[candidates.video_id == video_id]
        x_minutes = sal.timestamp.values / 60
        y_saliency = sal.saliency_smooth.values

        fig, (ax_video, ax_sal) = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={"height_ratios": [3, 2]})
        fig.patch.set_facecolor(SURFACE)

        first_frame = mpimg.imread(frame_paths[0])
        video_im = ax_video.imshow(first_frame)
        ax_video.axis("off")
        video_title = ax_video.set_title("", fontsize=11, weight="bold", color=INK)

        for _, seg in gt_segs.iterrows():
            ax_sal.axvspan(seg.start_ts / 60, seg.end_ts / 60, color=phase_color.get(seg.phase, MUTED),
                            alpha=0.12, linewidth=0)
        for _, clip in clips.iterrows():
            ax_sal.axvspan(clip.start_ts / 60, clip.end_ts / 60, color=GOOD, alpha=0.25, linewidth=0)
        ax_sal.plot(x_minutes, y_saliency, color=GRID, linewidth=1.1)  # full curve, faint, drawn once
        revealed_line, = ax_sal.plot([], [], color=INK, linewidth=1.6)  # left-to-right drawn-in portion
        playhead = ax_sal.axvline(0, color=CRITICAL, linewidth=1.4)
        ax_sal.axhline(operating_tau, color=CRITICAL, linestyle="--", linewidth=1.0, alpha=0.6)
        ax_sal.set_xlim(0, x_minutes.max())
        ax_sal.set_ylim(0, 1.05)
        ax_sal.set_xlabel("Time (min)")
        ax_sal.set_ylabel("Smoothed saliency")
        ax_sal.set_title(f"Video {video_id} — saliency over time", fontsize=10, weight="bold")
        fig.tight_layout()

        def update(i):
            frame = mpimg.imread(frame_paths[i])
            video_im.set_data(frame)
            ts = frame_timestamps[i]
            video_title.set_text(f"t={ts / 60:.1f} min")

            reveal_mask = sal.timestamp.values <= ts
            revealed_line.set_data(x_minutes[reveal_mask], y_saliency[reveal_mask])
            playhead.set_xdata([ts / 60, ts / 60])
            return video_im, revealed_line, playhead, video_title

        anim = animation.FuncAnimation(fig, update, frames=len(frame_paths), blit=False)
        writer = animation.FFMpegWriter(fps=target_fps, bitrate=4000)
        anim.save(str(out_path), writer=writer)
        plt.close(fig)
        print(f"Saved {out_path} ({len(frame_paths)} frames, {target_duration_s}s @ {target_fps}fps, "
              f"compressing {source_duration / 60:.0f} min of source video)")
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)


playthrough_path = GRAPH_DIRS["02b_saliency_playthrough"] / "saliency_playthrough_5.mp4"
render_saliency_playthrough("5", saliency, manifest, saliency_candidates,
                             CFG.saliency_operating_threshold, phase_color, playthrough_path,
                             target_duration_s=45, target_fps=15)
Video(str(playthrough_path), embed=True, width=700)



### 2.3 Per-phase rescue impact

How many (video, phase) pairs the global threshold left with at most one
candidate, and how many of those the per-phase rescue actually recovers.


In [ ]:

def rescue_impact_summary(saliency: pd.DataFrame, base_pool: pd.DataFrame, manifest: pd.DataFrame,
                           percentile: float) -> pd.DataFrame:
    table = pd.crosstab(base_pool.video_id, base_pool.phase)
    rows = []
    for video_id in sorted(saliency.video_id.unique(), key=lambda v: int(v)):
        vm = saliency[saliency.video_id == video_id]
        gt_phases = set(manifest[manifest.video_id == video_id].phase.unique())
        n_flagged, n_rescued = 0, 0
        for phase in gt_phases:
            count = table.loc[video_id, phase] if (video_id in table.index and phase in table.columns) else 0
            if count > 1:
                continue
            n_flagged += 1
            phase_frames = vm[vm.predicted_phase == phase]
            if len(phase_frames) == 0:
                continue
            cutoff = np.percentile(phase_frames.saliency_smooth, percentile)
            if (phase_frames.saliency_smooth >= cutoff).sum() > 0:
                n_rescued += 1
        rows.append({"video_id": video_id, "n_flagged": n_flagged, "n_rescued": n_rescued,
                      "n_still_failed": n_flagged - n_rescued})
    return pd.DataFrame(rows)


def plot_rescue_impact(impact: pd.DataFrame, out_path):
    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(impact))
    ax.bar(x, impact.n_rescued, color=GOOD, label="Rescued")
    ax.bar(x, impact.n_still_failed, bottom=impact.n_rescued, color=CRITICAL, label="Still underrepresented")
    ax.set_xticks(x)
    ax.set_xticklabels([f"Video {v}" for v in impact.video_id])
    ax.set_ylabel("(video, phase) pairs flagged as underrepresented")
    ax.set_title("Per-phase rescue: how many underrepresented phases it recovers", fontsize=11, weight="bold")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor=SURFACE)
    plt.show()
    print(f"Saved {out_path}")


rescue_impact = rescue_impact_summary(saliency, saliency_candidates_global, manifest,
                                       CFG.per_phase_rescue_percentile)
plot_rescue_impact(rescue_impact, GRAPH_DIRS["03_rescue_impact"] / "rescue_impact.png")
rescue_impact



### 2.4 Does the per-phase rescue actually buy coverage?

Global-threshold-only pool vs. the hybrid (threshold + rescue) pool,
compared on the notebook's own phase-coverage metric, per video.


In [ ]:

def plot_coverage_dumbbell(global_pool: pd.DataFrame, hybrid_pool: pd.DataFrame, manifest: pd.DataFrame, out_path):
    videos = sorted(manifest.video_id.unique(), key=lambda v: int(v))
    rows = []
    for video_id in videos:
        rows.append({
            "video_id": video_id,
            "global_cov": phase_coverage(global_pool[global_pool.video_id == video_id], manifest, video_id),
            "hybrid_cov": phase_coverage(hybrid_pool[hybrid_pool.video_id == video_id], manifest, video_id),
        })
    df = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(8, 5))
    y = np.arange(len(df))
    for i, row in df.iterrows():
        ax.plot([row.global_cov, row.hybrid_cov], [i, i], color=MUTED, linewidth=1.5, zorder=1)
    ax.scatter(df["global_cov"], y, color=PALETTE[1], s=90, label="Global threshold only", zorder=2)
    ax.scatter(df["hybrid_cov"], y, color=PALETTE[2], s=90, label="Hybrid (+ per-phase rescue)", zorder=2)
    ax.set_yticks(y)
    ax.set_yticklabels([f"Video {v}" for v in df.video_id])
    ax.set_xlabel("Phase coverage")
    ax.set_xlim(0, 1.05)
    ax.set_title("Does per-phase rescue actually buy coverage?", fontsize=11, weight="bold")
    ax.legend(frameon=False, loc="lower right")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor=SURFACE)
    plt.show()
    print(f"Saved {out_path}")
    return df


coverage_global_vs_hybrid = plot_coverage_dumbbell(
    saliency_candidates_global, saliency_candidates, manifest,
    GRAPH_DIRS["04_hybrid_vs_global_coverage"] / "coverage_dumbbell.png")
coverage_global_vs_hybrid



## 3. Reel comparison

### 3.1 Summary by variant — the headline chart

One panel per metric, all reel variants side by side. This alone tells
the whole "which strategy does what" story on a single slide.


In [ ]:

SUMMARY_METRICS = ["phase_coverage", "spearman_saliency_alignment",
                    "reel_to_video_cosine_similarity", "intra_reel_cosine_similarity"]
METRIC_LABEL = {
    "phase_coverage": "Phase coverage",
    "spearman_saliency_alignment": "Spearman saliency alignment",
    "reel_to_video_cosine_similarity": "Reel-to-video cosine similarity",
    "intra_reel_cosine_similarity": "Intra-reel cosine similarity",
}

summary_by_variant = metrics.groupby("reel_variant")[
    ["n_clips", "phase_coverage", "final_compression_ratio", "intra_reel_cosine_similarity",
     "reel_to_video_cosine_similarity", "spearman_saliency_alignment"]
].mean().reindex(VARIANT_ORDER)


def plot_summary_by_variant(summary_by_variant: pd.DataFrame, out_path):
    fig, axes = plt.subplots(1, len(SUMMARY_METRICS), figsize=(4.2 * len(SUMMARY_METRICS), 5), sharey=False)
    for ax, metric in zip(axes, SUMMARY_METRICS):
        values = summary_by_variant[metric]
        colors = [VARIANT_COLOR[v] for v in values.index]
        ax.bar(range(len(values)), values, color=colors)
        ax.set_xticks(range(len(values)))
        ax.set_xticklabels(values.index, rotation=45, ha="right", fontsize=8)
        ax.set_title(METRIC_LABEL[metric], fontsize=10, weight="bold")
        ax.axhline(0, color=GRID, linewidth=0.8)
    fig.suptitle("Summary by reel variant", fontsize=13, weight="bold")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor=SURFACE)
    plt.show()
    print(f"Saved {out_path}")


plot_summary_by_variant(summary_by_variant, GRAPH_DIRS["05_summary_by_variant"] / "summary_by_variant.png")
summary_by_variant.round(3)



### 3.2 Reel 3 vs. Reel 4 — per-video phase coverage

The direct paired comparison behind the 3-vs-4 decision.


In [ ]:

def plot_reel34_dumbbell(metrics: pd.DataFrame, reel3_variant: str, reel4_variant: str, out_path):
    pivot = metrics[metrics.reel_variant.isin([reel3_variant, reel4_variant])].pivot(
        index="video_id", columns="reel_variant", values="phase_coverage")
    pivot = pivot.reindex(sorted(pivot.index, key=int))

    fig, ax = plt.subplots(figsize=(8, 5))
    y = np.arange(len(pivot))
    for i, (_, row) in enumerate(pivot.iterrows()):
        ax.plot([row[reel3_variant], row[reel4_variant]], [i, i], color=MUTED, linewidth=1.5, zorder=1)
    ax.scatter(pivot[reel3_variant], y, color=VARIANT_COLOR[reel3_variant], s=90, label=reel3_variant, zorder=2)
    ax.scatter(pivot[reel4_variant], y, color=VARIANT_COLOR[reel4_variant], s=90, label=reel4_variant, zorder=2)
    ax.set_yticks(y)
    ax.set_yticklabels([f"Video {v}" for v in pivot.index])
    ax.set_xlabel("Phase coverage")
    ax.set_xlim(0, 1.05)
    ax.set_title("Reel 3 vs. Reel 4 — phase coverage per video", fontsize=11, weight="bold")
    ax.legend(frameon=False, loc="lower right")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor=SURFACE)
    plt.show()
    print(f"Saved {out_path}")
    return pivot


if REEL3_VARIANT and REEL4_VARIANT:
    reel34_pivot = plot_reel34_dumbbell(metrics, REEL3_VARIANT, REEL4_VARIANT,
                                         GRAPH_DIRS["06_reel3_vs_reel4_dumbbell"] / "reel3_vs_reel4_phase_coverage.png")
else:
    reel34_pivot = pd.DataFrame()
    print("Reel 3 and/or Reel 4 selections not found on disk — skipping.")
reel34_pivot



### 3.3 Reel 3 vs. Reel 4 — head-to-head win counts

For each quality metric, how many videos each variant wins. Compression
ratio and clip count are excluded — they're design targets, not quality
signals, so "winning" them isn't meaningful.


In [ ]:

HIGHER_IS_BETTER = {
    "phase_coverage": True,
    "spearman_saliency_alignment": True,
    "reel_to_video_cosine_similarity": True,
    "intra_reel_cosine_similarity": False,  # lower = less redundant
}


def head_to_head_counts(metrics: pd.DataFrame, variant_a: str, variant_b: str) -> pd.DataFrame:
    rows = []
    for metric, higher_better in HIGHER_IS_BETTER.items():
        pivot = metrics[metrics.reel_variant.isin([variant_a, variant_b])].pivot(
            index="video_id", columns="reel_variant", values=metric).dropna()
        if higher_better:
            a_wins = (pivot[variant_a] > pivot[variant_b]).sum()
            b_wins = (pivot[variant_b] > pivot[variant_a]).sum()
        else:
            a_wins = (pivot[variant_a] < pivot[variant_b]).sum()
            b_wins = (pivot[variant_b] < pivot[variant_a]).sum()
        ties = len(pivot) - a_wins - b_wins
        rows.append({"metric": METRIC_LABEL[metric], "a_wins": a_wins, "b_wins": b_wins,
                      "ties": ties, "n_videos": len(pivot)})
    return pd.DataFrame(rows)


def plot_head_to_head(counts: pd.DataFrame, variant_a: str, variant_b: str, out_path):
    fig, ax = plt.subplots(figsize=(9, 5))
    y = np.arange(len(counts))
    ax.barh(y, counts.a_wins, color=VARIANT_COLOR[variant_a], label=f"{variant_a} wins")
    ax.barh(y, counts.b_wins, left=counts.a_wins, color=VARIANT_COLOR[variant_b], label=f"{variant_b} wins")
    ax.barh(y, counts.ties, left=counts.a_wins + counts.b_wins, color=GRID, label="Tie")
    ax.set_yticks(y)
    ax.set_yticklabels(counts.metric)
    ax.set_xlabel("Number of videos")
    ax.set_title(f"{variant_a} vs. {variant_b} — head-to-head by metric", fontsize=11, weight="bold")
    ax.legend(frameon=False, loc="lower right")
    ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor=SURFACE)
    plt.show()
    print(f"Saved {out_path}")


if REEL3_VARIANT and REEL4_VARIANT:
    head_to_head = head_to_head_counts(metrics, REEL3_VARIANT, REEL4_VARIANT)
    plot_head_to_head(head_to_head, REEL3_VARIANT, REEL4_VARIANT,
                       GRAPH_DIRS["07_reel3_vs_reel4_head_to_head"] / "head_to_head.png")
else:
    head_to_head = pd.DataFrame()
    print("Reel 3 and/or Reel 4 selections not found on disk — skipping.")
head_to_head



### 3.4 Phase-coverage checklist (temporal, not label-based)

For each ground-truth phase and each reel variant: the fraction of that
phase's own frames actually covered by a selected clip, averaged over
every video where the phase occurs. This is the true frame-level overlap
metric discussed earlier — a stricter, more honest check than "does the
reel contain a clip whose label matches this phase".


In [ ]:

def build_phase_checklist(all_reels: dict, manifest: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (video_id, variant), clips in all_reels.items():
        coverage = temporal_coverage_per_phase(clips, manifest, video_id)
        for phase, frac in coverage.items():
            rows.append({"video_id": video_id, "reel_variant": variant, "phase": phase, "coverage_frac": frac})
    return pd.DataFrame(rows)


def plot_phase_checklist(checklist: pd.DataFrame, phase_order: list, variant_order: list, out_path):
    pivot = checklist.pivot_table(index="phase", columns="reel_variant", values="coverage_frac", aggfunc="mean")
    pivot = pivot.reindex(index=[p for p in phase_order if p in pivot.index],
                            columns=[v for v in variant_order if v in pivot.columns])

    fig, ax = plt.subplots(figsize=(1.6 * len(pivot.columns) + 3, 0.42 * len(pivot.index) + 2))
    im = ax.imshow(pivot.values, cmap=BLUE_SEQUENTIAL, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            if np.isnan(val):
                continue
            text_color = "white" if val > 0.6 else INK
            ax.text(j, i, f"{val:.0%}", ha="center", va="center", fontsize=7.5, color=text_color)
    ax.set_title("Phase-coverage checklist — mean temporal overlap per phase", fontsize=11, weight="bold")
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="Mean coverage fraction")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor=SURFACE)
    plt.show()
    print(f"Saved {out_path}")
    return pivot


phase_checklist = build_phase_checklist(all_reels, manifest)
phase_order = get_phase_order(manifest)
phase_checklist_pivot = plot_phase_checklist(
    phase_checklist, phase_order, VARIANT_ORDER,
    GRAPH_DIRS["08_phase_coverage_checklist"] / "phase_coverage_checklist.png")
phase_checklist_pivot.round(2)



### 3.5 Best reel per video

Every reel variant's phase coverage, grouped by video — the tallest bar
in each group is that video's winner, and the full ranking (not just the
winner) is visible at a glance.


In [ ]:

def plot_best_reel_ranking(metrics: pd.DataFrame, variant_order: list, out_path):
    pivot = metrics.pivot(index="video_id", columns="reel_variant", values="phase_coverage")
    pivot = pivot.reindex(index=sorted(pivot.index, key=int),
                            columns=[v for v in variant_order if v in pivot.columns])

    n_variants = len(pivot.columns)
    width = 0.8 / n_variants
    x = np.arange(len(pivot.index))

    fig, ax = plt.subplots(figsize=(11, 5.5))
    for i, variant in enumerate(pivot.columns):
        ax.bar(x + i * width, pivot[variant], width=width, color=VARIANT_COLOR[variant], label=variant)
    ax.set_xticks(x + width * (n_variants - 1) / 2)
    ax.set_xticklabels([f"Video {v}" for v in pivot.index])
    ax.set_ylabel("Phase coverage")
    ax.set_title("Best reel per video — full ranking, not just the winner", fontsize=11, weight="bold")
    ax.legend(frameon=False, ncol=n_variants, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12))
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor=SURFACE)
    plt.show()
    print(f"Saved {out_path}")
    return pivot


best_reel_pivot = plot_best_reel_ranking(metrics, VARIANT_ORDER,
                                          GRAPH_DIRS["09_best_reel_ranking"] / "best_reel_ranking.png")
best_reel_winners = best_reel_pivot.idxmax(axis=1).rename("best_variant").to_frame()
best_reel_winners["phase_coverage"] = best_reel_pivot.max(axis=1)
best_reel_winners



### 3.6 Inter-reel Jaccard overlap

How similar each pair of reel strategies' actual frame selections are,
averaged across every video where both variants ran — low overlap between
two variants means they're genuinely picking different footage, not just
re-deriving the same answer two ways.


In [ ]:

def load_jaccard(video_id: str) -> pd.DataFrame:
    path = DIRS["metrics"] / f"jaccard_{video_id}.csv"
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, index_col=0)


def average_jaccard_matrix(videos: list) -> pd.DataFrame:
    pair_values = {}
    for video_id in videos:
        matrix = load_jaccard(video_id)
        if matrix.empty:
            continue
        for a in matrix.index:
            for b in matrix.columns:
                if a >= b:
                    continue
                val = matrix.loc[a, b]
                if pd.isna(val):
                    continue
                pair_values.setdefault((a, b), []).append(val)
    variants = sorted({v for pair in pair_values for v in pair})
    out = pd.DataFrame(np.nan, index=variants, columns=variants)
    for (a, b), values in pair_values.items():
        out.loc[a, b] = out.loc[b, a] = np.mean(values)
    np.fill_diagonal(out.values, 1.0)
    return out


def plot_jaccard_heatmap(matrix: pd.DataFrame, out_path):
    fig, ax = plt.subplots(figsize=(1.3 * len(matrix) + 2, 1.1 * len(matrix) + 2))
    im = ax.imshow(matrix.values, cmap=BLUE_SEQUENTIAL, vmin=0, vmax=1)
    ax.set_xticks(range(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(matrix.index)))
    ax.set_yticklabels(matrix.index, fontsize=8)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix.values[i, j]
            if np.isnan(val):
                continue
            text_color = "white" if val > 0.6 else INK
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, color=text_color)
    ax.set_title("Mean inter-reel Jaccard overlap (across videos)", fontsize=11, weight="bold")
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor=SURFACE)
    plt.show()
    print(f"Saved {out_path}")


jaccard_avg = average_jaccard_matrix(CFG.videos)
if not jaccard_avg.empty:
    plot_jaccard_heatmap(jaccard_avg, GRAPH_DIRS["10_inter_reel_jaccard"] / "mean_jaccard_overlap.png")
else:
    print("No jaccard_*.csv files found — run run_all_jaccard() in stage4_highlight_reels.ipynb first.")
jaccard_avg.round(2)



## 4. Tables document

Every table used above, written into one Word document so it can be
dropped straight into a report or appendix.


In [ ]:

INK_HEX, MUTED_HEX = "0B0B0B", "52514E"


def add_heading(doc, text, level=1):
    doc.add_heading(text, level=level)


def add_dataframe_table(doc: Document, df: pd.DataFrame, include_index: bool = True, index_label: str = ""):
    display_df = df.reset_index() if include_index else df.copy()
    if include_index and index_label:
        display_df = display_df.rename(columns={display_df.columns[0]: index_label})

    table = doc.add_table(rows=1, cols=len(display_df.columns))
    table.style = "Light Grid Accent 1"
    header_cells = table.rows[0].cells
    for j, col in enumerate(display_df.columns):
        header_cells[j].text = str(col)
        for p in header_cells[j].paragraphs:
            for r in p.runs:
                r.bold = True

    for _, row in display_df.iterrows():
        cells = table.add_row().cells
        for j, col in enumerate(display_df.columns):
            val = row[col]
            cells[j].text = f"{val:.3f}" if isinstance(val, float) else str(val)
    doc.add_paragraph()


doc = Document()
doc.styles["Normal"].font.name = "Calibri"
doc.styles["Normal"].font.size = Pt(10)

title = doc.add_heading("Stage 4 — Summary Tables", level=0)

add_heading(doc, "Threshold sweep (segment count by video x tau)")
add_dataframe_table(doc, threshold_sweep.pivot(index="video_id", columns="threshold", values="n_segments"),
                     index_label="video_id")

add_heading(doc, "Per-phase rescue impact")
add_dataframe_table(doc, rescue_impact, include_index=False)

add_heading(doc, "Global vs. hybrid pool — phase coverage per video")
add_dataframe_table(doc, coverage_global_vs_hybrid, include_index=False)

add_heading(doc, "Summary by reel variant")
add_dataframe_table(doc, summary_by_variant.round(3), index_label="reel_variant")

if not reel34_pivot.empty:
    add_heading(doc, "Reel 3 vs. Reel 4 — phase coverage per video")
    add_dataframe_table(doc, reel34_pivot.round(3), index_label="video_id")

if not head_to_head.empty:
    add_heading(doc, "Reel 3 vs. Reel 4 — head-to-head win counts")
    add_dataframe_table(doc, head_to_head, include_index=False)

add_heading(doc, "Phase-coverage checklist (mean temporal overlap)")
add_dataframe_table(doc, phase_checklist_pivot.round(2), index_label="phase")

add_heading(doc, "Best reel per video")
add_dataframe_table(doc, best_reel_winners.round(3), index_label="video_id")

if not jaccard_avg.empty:
    add_heading(doc, "Mean inter-reel Jaccard overlap")
    add_dataframe_table(doc, jaccard_avg.round(3), index_label="reel_variant")

doc_path = GRAPH_DIRS["tables"] / "stage4_summary_tables.docx"
doc.save(doc_path)
print(f"Saved {doc_path}")



Optional — download the tables document directly from Colab:


In [ ]:

try:
    from google.colab import files
    files.download(str(doc_path))
except ImportError:
    print("Not running in Colab — find the file on Drive at:", doc_path)
